# 진짜 정규화용 실행 모음집

1. 필수 수집 : brand_code, name, english_name
2. 필수 추가 작업 : 카테고리, logo_url, 가격대,product_categories, target_gender, target_age
3. 부가적 : country_code
4. 

## 1. 있는 브랜드 매핑시키기

In [ ]:
from tools.brand_mapping.mapper import BrandMapper
from core.models import BrandSource


mapper = BrandMapper(
    enable_contains=True,
    enable_similarity=True,
    similarity_threshold=0.88,
)


qs = (
    BrandSource.objects
    .filter(
        brand__isnull=True,
        mapping_status=BrandSource.MappingStatus.UNMAPPED,
    )
    .select_related("source")
    .order_by(
        "source_id",
        "-detected_count",
        "id",
    )
)


total = qs.count()

success = 0
skipped = 0
failed = 0


print("=" * 110)
print("BrandSource EXACT 자동 매핑 시작")
print("=" * 110)
print("전체 미매핑 대상:", total)
print()


for index, bs in enumerate(
    qs.iterator(),
    start=1,
):

    try:
        result = mapper.match(bs)

        # EXACT만 허용
        if (
            result.brand_id is None
            or result.method
            != BrandSource.MappingMethod.EXACT_NAME
        ):
            skipped += 1
            continue

        # 실제 DB 반영
        applied = mapper.apply(bs)

        success += 1

        print(
            f"[{index}/{total}] EXACT | "
            f"{bs.source.code} | "
            f"BS#{bs.id} "
            f"{bs.name} / {bs.english_name} "
            f"-> "
            f"Brand#{applied.brand_id} "
            f"{applied.brand_name} / "
            f"{applied.brand_english_name}"
        )

    except Exception as exc:
        failed += 1

        print(
            f"[{index}/{total}] FAILED | "
            f"BS#{bs.id} "
            f"{bs.name} | "
            f"{type(exc).__name__}: {exc}"
        )


print()
print("=" * 110)
print("EXACT 자동 매핑 완료")
print("=" * 110)

print("전체 미매핑 :", total)
print("EXACT 매핑  :", success)
print("SKIP        :", skipped)
print("FAILED      :", failed)

print("=" * 110)

## 2. 남아있는 무신사 미매핑 브랜드 소스들 전부 브랜드로 신규 승격

In [ ]:
from django.db import transaction

from core.models import BrandSource
from core.models import Brand


SOURCE_CODE = "musinsa"


qs = (
    BrandSource.objects
    .filter(
        source__code__iexact=SOURCE_CODE,
        brand__isnull=True,
        mapping_status=BrandSource.MappingStatus.UNMAPPED,
    )
    .select_related("source")
    .order_by(
        "-detected_count",
        "id",
    )
)


total = qs.count()

print("=" * 110)
print("무신사 미매핑 BrandSource -> FEEDIT Brand 신규 승격")
print("=" * 110)
print("승격 대상:", total)
print()


created_count = 0
failed_count = 0


for index, bs in enumerate(qs.iterator(), start=1):

    try:
        with transaction.atomic():

            # --------------------------------------------------
            # Brand code
            # --------------------------------------------------

            source_id = str(bs.source_brand_id).strip()

            brand_code = (
                f"BRAND_MUSINSA_{source_id}"
                .upper()
            )


            # --------------------------------------------------
            # Brand 생성
            # --------------------------------------------------

            brand = Brand.objects.create(
                brand_code=brand_code,

                name=(
                    bs.name.strip()
                    if bs.name
                    else source_id
                ),

                english_name=(
                    bs.english_name.strip().upper()
                    if bs.english_name
                    else source_id.upper()
                ),

                image_url=bs.image_url,
                country_code=bs.country_code,
                description=bs.description,
                target_gender=bs.target_gender,
                target_age=bs.target_age,
                website_url=bs.website_url,

                status=Brand.Status.ACTIVE,
            )


            # --------------------------------------------------
            # BrandSource -> Brand 연결
            # --------------------------------------------------

            bs.brand = brand

            bs.mapping_status = (
                BrandSource.MappingStatus.MANUAL_MAPPED
            )

            bs.mapping_method = (
                BrandSource.MappingMethod.MANUAL
            )

            bs.mapping_confidence = 1.0

            bs.save(
                update_fields=[
                    "brand",
                    "mapping_status",
                    "mapping_method",
                    "mapping_confidence",
                    "updated_at",
                ]
            )


            created_count += 1


            print(
                f"[{index}/{total}] CREATED | "
                f"발견={bs.detected_count or 0:<6} | "
                f"BS#{bs.id:<6} | "
                f"{bs.name} / {bs.english_name} "
                f"-> "
                f"Brand#{brand.id} "
                f"[{brand.brand_code}] "
                f"{brand.name} / {brand.english_name}"
            )


    except Exception as exc:

        failed_count += 1

        print(
            f"[{index}/{total}] FAILED | "
            f"BS#{bs.id} | "
            f"{bs.name} | "
            f"{type(exc).__name__}: {exc}"
        )


print()
print("=" * 110)
print("무신사 미매핑 브랜드 승격 완료")
print("=" * 110)

print("대상 :", total)
print("생성 :", created_count)
print("실패 :", failed_count)

print("=" * 110)

## 3. 무신사 승격 후 유즈드 작업 진행 (2순위)

- 즉 브랜드록 승격할 만큼 프로필이 갖춰진 애들만
- name, english_name, image_url, description 가 채워진 애들 중에서 발견 횟수가 높은 순으로 승격

### 3.1 일단 EXACT 승격

In [ ]:
from tools.brand_mapping.mapper import BrandMapper
from core.models import BrandSource


SOURCE_CODE = "musinsa_used"


mapper = BrandMapper(
    enable_contains=True,
    enable_similarity=True,
    similarity_threshold=0.88,
)


qs = (
    BrandSource.objects
    .filter(
        source__code__iexact=SOURCE_CODE,
        brand__isnull=True,
        mapping_status=BrandSource.MappingStatus.UNMAPPED,
    )
    .select_related("source")
    .order_by(
        "-detected_count",
        "id",
    )
)


total = qs.count()

success = 0
skipped = 0
failed = 0


print("=" * 110)
print("무신사 USED 미매핑 -> EXACT 재매핑")
print("=" * 110)
print("전체 대상:", total)
print()


for index, bs in enumerate(qs.iterator(), start=1):

    try:
        result = mapper.match(bs)

        # EXACT_NAME만 실제 반영
        if (
            result.brand_id is None
            or result.method
            != BrandSource.MappingMethod.EXACT_NAME
        ):
            skipped += 1
            continue

        applied = mapper.apply(bs)

        success += 1

        print(
            f"[{index}/{total}] EXACT | "
            f"발견={bs.detected_count or 0:<6} | "
            f"BS#{bs.id:<6} | "
            f"{bs.name} / {bs.english_name} "
            f"-> "
            f"Brand#{applied.brand_id} "
            f"{applied.brand_name} / "
            f"{applied.brand_english_name}"
        )

    except Exception as exc:

        failed += 1

        print(
            f"[{index}/{total}] FAILED | "
            f"BS#{bs.id} | "
            f"{bs.name} | "
            f"{type(exc).__name__}: {exc}"
        )


print()
print("=" * 110)
print("무신사 USED EXACT 재매핑 완료")
print("=" * 110)
print("전체 미매핑 :", total)
print("EXACT 매핑  :", success)
print("EXACT 아님  :", skipped)
print("실패        :", failed)
print("=" * 110)

### 3.2 다음 브랜드 정보가 채워진 브랜드소스들만 승격

In [ ]:
from django.db import transaction
from django.db.models import Q

from core.models import Brand, BrandSource


SOURCE_CODE = "musinsa_used"


# ============================================================
# 승격 가능한 USED BrandSource
# ============================================================

qs = (
    BrandSource.objects
    .filter(
        source__code__iexact=SOURCE_CODE,
        brand__isnull=True,
        mapping_status=BrandSource.MappingStatus.UNMAPPED,
    )
    .exclude(
        Q(name__isnull=True)
        | Q(name__exact="")
        | Q(english_name__isnull=True)
        | Q(english_name__exact="")
        | Q(image_url__isnull=True)
        | Q(image_url__exact="")
        | Q(description__isnull=True)
        | Q(description__exact="")
    )
    .select_related("source")
    .order_by(
        "-detected_count",
        "id",
    )
)


total = qs.count()


print("=" * 110)
print("무신사 USED 신규 Brand 승격")
print("=" * 110)
print("필수 조건:")
print("- brand name")
print("- english name")
print("- logo")
print("- description")
print()
print("승격 가능:", total)
print("=" * 110)
print()


created_count = 0
failed_count = 0


for index, bs in enumerate(qs.iterator(), start=1):

    try:

        with transaction.atomic():

            source_brand_id = str(
                bs.source_brand_id
            ).strip()

            # USED suffix 제거
            clean_source_id = source_brand_id

            if clean_source_id.lower().endswith("_used"):
                clean_source_id = clean_source_id[:-5]


            # --------------------------------------------------
            # Brand code
            # --------------------------------------------------

            brand_code = (
                f"BRAND_{clean_source_id}"
                .upper()
            )


            # --------------------------------------------------
            # 혹시 같은 brand_code가 이미 존재하면 생성 금지
            # --------------------------------------------------

            if Brand.objects.filter(
                brand_code=brand_code
            ).exists():

                print(
                    f"[{index}/{total}] SKIP | "
                    f"발견={bs.detected_count or 0:<6} | "
                    f"BS#{bs.id:<6} | "
                    f"{bs.name} | "
                    f"이미 존재: {brand_code}"
                )

                continue


            # --------------------------------------------------
            # Brand 신규 생성
            # --------------------------------------------------

            brand = Brand.objects.create(
                brand_code=brand_code,

                name=bs.name.strip(),

                english_name=(
                    bs.english_name
                    .strip()
                    .upper()
                ),

                image_url=bs.image_url,

                description=(
                    bs.description.strip()
                ),

                country_code=bs.country_code,

                target_gender=bs.target_gender,

                target_age=bs.target_age,

                website_url=bs.website_url,

                status=Brand.Status.ACTIVE,
            )


            # --------------------------------------------------
            # BrandSource -> 신규 Brand 연결
            # --------------------------------------------------

            bs.brand = brand

            bs.mapping_status = (
                BrandSource.MappingStatus.MANUAL_MAPPED
            )

            bs.mapping_method = (
                BrandSource.MappingMethod.MANUAL
            )

            bs.mapping_confidence = 1.0

            bs.save(
                update_fields=[
                    "brand",
                    "mapping_status",
                    "mapping_method",
                    "mapping_confidence",
                    "updated_at",
                ]
            )


            created_count += 1


            print(
                f"[{index}/{total}] CREATED | "
                f"발견={bs.detected_count or 0:<6} | "
                f"BS#{bs.id:<6} | "
                f"{bs.name} / "
                f"{bs.english_name} "
                f"-> "
                f"Brand#{brand.id} | "
                f"{brand.brand_code}"
            )


    except Exception as exc:

        failed_count += 1

        print(
            f"[{index}/{total}] FAILED | "
            f"BS#{bs.id} | "
            f"{bs.name} | "
            f"{type(exc).__name__}: {exc}"
        )


print()
print("=" * 110)
print("무신사 USED 신규 Brand 승격 완료")
print("=" * 110)
print("승격 가능 :", total)
print("신규 생성 :", created_count)
print("실패      :", failed_count)
print("=" * 110)

## 4. 지그재그 미매핑 브랜드 작업
1.

### 4.1 일단 EXACT 승격

In [ ]:
from tools.brand_mapping.mapper import BrandMapper
from core.models import BrandSource


SOURCE_CODE = "zigzag"


mapper = BrandMapper(
    enable_contains=True,
    enable_similarity=True,
    similarity_threshold=0.88,
)


qs = (
    BrandSource.objects
    .filter(
        source__code__iexact=SOURCE_CODE,
        brand__isnull=True,
        mapping_status=BrandSource.MappingStatus.UNMAPPED,
    )
    .select_related("source")
    .order_by(
        "-detected_count",
        "id",
    )
)


total = qs.count()

success = 0
skipped = 0
failed = 0


print("=" * 110)
print("지그재그 미매핑 -> EXACT 자동 매핑")
print("=" * 110)
print("전체 대상:", total)
print()


for index, bs in enumerate(qs.iterator(), start=1):

    try:
        result = mapper.match(bs)

        # EXACT만 반영
        if (
            result.brand_id is None
            or result.method
            != BrandSource.MappingMethod.EXACT_NAME
        ):
            skipped += 1
            continue

        applied = mapper.apply(bs)

        success += 1

        print(
            f"[{index}/{total}] EXACT | "
            f"발견={bs.detected_count or 0:<6} | "
            f"BS#{bs.id:<6} | "
            f"{bs.name} / {bs.english_name} "
            f"-> "
            f"Brand#{applied.brand_id} "
            f"{applied.brand_name} / "
            f"{applied.brand_english_name}"
        )

    except Exception as exc:
        failed += 1

        print(
            f"[{index}/{total}] FAILED | "
            f"BS#{bs.id} | "
            f"{bs.name} | "
            f"{type(exc).__name__}: {exc}"
        )


print()
print("=" * 110)
print("지그재그 EXACT 자동 매핑 완료")
print("=" * 110)
print("전체 대상 :", total)
print("EXACT 매핑 :", success)
print("EXACT 아님 :", skipped)
print("실패       :", failed)
print("=" * 110)

### 4.2 Zigzag 발견 10회 이상 + 필수 프로필 존재 → 신규 Brand
- 지그재그 기준 필수 프로필 : name , english_name, image_url    

In [ ]:
import re

from django.db import transaction
from django.db.models import Q

from core.models import Brand, BrandSource


SOURCE_CODE = "zigzag"
MIN_DETECTED_COUNT = 10


# ============================================================
# Brand Code
# BRAND_{ENGLISH_NAME}
# ============================================================

def make_brand_code(english_name: str) -> str:
    normalized = english_name.strip().upper()

    # 공백 / 하이픈 / 특수문자 덩어리 -> 언더바
    normalized = re.sub(
        r"[^A-Z0-9]+",
        "_",
        normalized,
    )

    # 앞뒤 언더바 제거
    normalized = normalized.strip("_")

    # 연속 언더바 정리
    normalized = re.sub(
        r"_+",
        "_",
        normalized,
    )

    if not normalized:
        raise ValueError(
            f"brand_code 생성 불가: {english_name!r}"
        )

    return f"BRAND_{normalized}"


# ============================================================
# 승격 대상
# ============================================================

qs = (
    BrandSource.objects
    .filter(
        source__code__iexact=SOURCE_CODE,
        brand__isnull=True,
        mapping_status=BrandSource.MappingStatus.UNMAPPED,
        detected_count__gte=MIN_DETECTED_COUNT,
    )
    .exclude(
        Q(name__isnull=True)
        | Q(name__exact="")
        | Q(english_name__isnull=True)
        | Q(english_name__exact="")
        | Q(image_url__isnull=True)
        | Q(image_url__exact="")
    )
    .select_related("source")
    .order_by(
        "-detected_count",
        "id",
    )
)


total = qs.count()

created_count = 0
linked_count = 0
failed_count = 0


print("=" * 120)
print("지그재그 미매핑 -> FEEDIT Brand 신규 승격")
print("=" * 120)

print("조건")
print(f"- detected_count >= {MIN_DETECTED_COUNT}")
print("- name 필수")
print("- english_name 필수")
print("- image_url 필수")
print("- description 선택")
print("- brand_code = BRAND_{ENGLISH_NAME}")

print()
print("승격 대상:", total)
print("=" * 120)
print()


# ============================================================
# 실행
# ============================================================

for index, bs in enumerate(qs.iterator(), start=1):

    try:
        with transaction.atomic():

            # --------------------------------------------------
            # 표준 이름
            # --------------------------------------------------

            brand_name = bs.name.strip()

            english_name = (
                bs.english_name
                .strip()
                .upper()
            )

            brand_code = make_brand_code(
                english_name
            )


            # --------------------------------------------------
            # 동일 Brand Code 존재 확인
            # --------------------------------------------------

            existing_brand = (
                Brand.objects
                .filter(
                    brand_code=brand_code
                )
                .first()
            )


            # --------------------------------------------------
            # 이미 존재하면 새로 만들지 않고 연결
            # --------------------------------------------------

            if existing_brand:

                bs.brand = existing_brand

                bs.mapping_status = (
                    BrandSource.MappingStatus.MANUAL_MAPPED
                )

                bs.mapping_method = (
                    BrandSource.MappingMethod.MANUAL
                )

                bs.mapping_confidence = 1.0

                bs.save(
                    update_fields=[
                        "brand",
                        "mapping_status",
                        "mapping_method",
                        "mapping_confidence",
                        "updated_at",
                    ]
                )

                linked_count += 1

                print(
                    f"[{index}/{total}] LINKED  | "
                    f"발견={bs.detected_count or 0:<6} | "
                    f"BS#{bs.id:<6} | "
                    f"{brand_name} / {english_name} "
                    f"-> "
                    f"기존 Brand#{existing_brand.id} | "
                    f"{existing_brand.brand_code}"
                )

                continue


            # --------------------------------------------------
            # 신규 FEEDIT Brand 생성
            # --------------------------------------------------

            brand = Brand.objects.create(
                brand_code=brand_code,

                name=brand_name,

                english_name=english_name,

                image_url=bs.image_url,

                description=(
                    bs.description.strip()
                    if bs.description
                    else None
                ),

                country_code=bs.country_code,

                target_gender=bs.target_gender,

                target_age=bs.target_age,

                website_url=bs.website_url,

                status=Brand.Status.ACTIVE,
            )


            # --------------------------------------------------
            # BrandSource -> Brand 연결
            # --------------------------------------------------

            bs.brand = brand

            bs.mapping_status = (
                BrandSource.MappingStatus.MANUAL_MAPPED
            )

            bs.mapping_method = (
                BrandSource.MappingMethod.MANUAL
            )

            bs.mapping_confidence = 1.0

            bs.save(
                update_fields=[
                    "brand",
                    "mapping_status",
                    "mapping_method",
                    "mapping_confidence",
                    "updated_at",
                ]
            )


            created_count += 1


            print(
                f"[{index}/{total}] CREATED | "
                f"발견={bs.detected_count or 0:<6} | "
                f"BS#{bs.id:<6} | "
                f"{brand_name} / {english_name} "
                f"-> "
                f"Brand#{brand.id} | "
                f"{brand.brand_code}"
            )


    except Exception as exc:

        failed_count += 1

        print(
            f"[{index}/{total}] FAILED  | "
            f"발견={bs.detected_count or 0:<6} | "
            f"BS#{bs.id:<6} | "
            f"{bs.name} / {bs.english_name} | "
            f"{type(exc).__name__}: {exc}"
        )


# ============================================================
# 결과
# ============================================================

print()
print("=" * 120)
print("지그재그 신규 Brand 승격 완료")
print("=" * 120)

print("승격 대상 :", total)
print("신규 생성 :", created_count)
print("기존 연결 :", linked_count)
print("실패      :", failed_count)

print("=" * 120)